# 🔬 DeepTrace v2 — EfficientNet-B4 Training Pipeline

This notebook trains an **EfficientNet-B4** model for facial deepfake detection. It implements the two critical requirements for our forensic pipeline:
1. **MTCNN Face Cropping**: Extracts faces from raw images with a 20% margin (exactly matching our FastAPI backend).
2. **Forensic Augmentations**: Simulates social media degradation (JPEG compression, blur, noise) using Albumentations to prevent real-world false negatives.

### ⚡ Prerequisites:
Set your Colab Runtime to **T4 GPU** or **A100 GPU** (`Runtime > Change runtime type`).

In [ ]:
# Step 1: Install Required Dependencies
!pip install -q transformers datasets accelerate torchvision evaluate kagglehub scikit-learn albumentations facenet-pytorch

## 1. Dataset Preparation & MTCNN Face Extraction
DeepTrace analyzes *faces*, not full images. We need to extract the faces from our training data before feeding them to the model.

In [ ]:
import os
import cv2
from PIL import Image
import numpy as np
import torch
from facenet_pytorch import MTCNN
import kagglehub
from tqdm.notebook import tqdm

# 1. Download a raw face/deepfake dataset (e.g., from Kaggle)
# Note: For a production model, you should upload FaceForensics++ to your Google Drive.
# For this template, we'll download a sample deepfake dataset.
print("Downloading dataset...")
dataset_path = kagglehub.dataset_download("xhlulu/140k-real-and-fake-faces") 
print(f"Raw dataset downloaded to: {dataset_path}")

# 2. Initialize MTCNN (exactly matching analyzer.py)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
mtcnn = MTCNN(keep_all=True, min_face_size=20, thresholds=[0.5, 0.6, 0.6], device=device)

def extract_face_from_image(img_path, save_path):
    """Extracts the face with a 20% margin, matching our inference pipeline."""
    try:
        image = Image.open(img_path).convert('RGB')
        # Scale down if too large
        max_dim = max(image.size)
        scale_factor = 1.0
        if max_dim > 1920:
            scale_factor = 1920.0 / max_dim
            new_size = (int(image.width * scale_factor), int(image.height * scale_factor))
            detect_img = image.resize(new_size, Image.Resampling.LANCZOS)
        else:
            detect_img = image

        boxes, probs = mtcnn.detect(np.array(detect_img))
        if boxes is not None and len(boxes) > 0:
            # Take the most confident face
            box = boxes[0]
            prob = probs[0]
            if prob > 0.60:
                x1, y1, x2, y2 = [int(c / scale_factor) for c in box]
                w, h = x2 - x1, y2 - y1
                # 20% margin
                cx1 = max(0, int(x1 - w * 0.2))
                cy1 = max(0, int(y1 - h * 0.2))
                cx2 = min(image.width, int(x2 + w * 0.2))
                cy2 = min(image.height, int(y2 + h * 0.2))
                
                face_crop = image.crop((cx1, cy1, cx2, cy2))
                face_crop.save(save_path, quality=95)
                return True
    except Exception as e:
        pass
    return False

# Note: We are simulating the cropping step here. 
# In a real scenario, you'd iterate over your raw dataset and save the crops to a new directory.
# For this notebook, we'll load the dataset directly assuming it contains faces.
print("MTCNN Face Extractor initialized.")

## 2. Aggressive Forensic Augmentations
To survive WhatsApp and Instagram compression, we *must* simulate it during training using `albumentations`.

In [ ]:
import albumentations as A
from albumentations.pytorch import ToTensorV2
from datasets import load_dataset
import numpy as np

# EfficientNet-B4 native resolution is 380x380
IMAGE_SIZE = 380

# The critical augmentation stack
train_augmentations = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.05, rotate_limit=15, p=0.5),
    
    # Color & Lighting
    A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1, p=0.5),
    
    # ⚡ CRITICAL: Social Media Compression Simulation ⚡
    A.ImageCompression(quality_lower=30, quality_upper=90, p=0.6),
    A.GaussianBlur(blur_limit=(3, 7), p=0.3),
    A.GaussNoise(var_limit=(10.0, 50.0), p=0.3),
    
    # Spatial 
    A.RandomResizedCrop(height=IMAGE_SIZE, width=IMAGE_SIZE, scale=(0.8, 1.0)),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

val_augmentations = A.Compose([
    A.Resize(height=IMAGE_SIZE, width=IMAGE_SIZE),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

def apply_train_transforms(examples):
    # Albumentations expects numpy arrays (H, W, C)
    images = [np.array(img.convert("RGB")) for img in examples["image"]]
    examples["pixel_values"] = [
        train_augmentations(image=img)["image"] for img in images
    ]
    # Convert to standard PyTorch tensor format (C, H, W)
    examples["pixel_values"] = [torch.tensor(img).permute(2, 0, 1) for img in examples["pixel_values"]]
    return examples

def apply_val_transforms(examples):
    images = [np.array(img.convert("RGB")) for img in examples["image"]]
    examples["pixel_values"] = [
        val_augmentations(image=img)["image"] for img in images
    ]
    examples["pixel_values"] = [torch.tensor(img).permute(2, 0, 1) for img in examples["pixel_values"]]
    return examples

print("Augmentation pipeline initialized.")

In [ ]:
# Load Dataset (Replace with your MTCNN-cropped dataset path)
# Here we use a generic deepfake face dataset for demonstration
print("Loading dataset...")
dataset = load_dataset("imagefolder", data_dir=dataset_path + "/real_vs_fake/real-vs-fake")

# Map classes explicitly to avoid silent inversion
class_names = dataset["train"].features["label"].names
id2label = {i: name.capitalize() for i, name in enumerate(class_names)}
label2id = {name.capitalize(): i for i, name in enumerate(class_names)}
print(f"Label Mapping: {id2label}")

# Apply transforms
train_ds = dataset["train"].with_transform(apply_train_transforms)
val_ds = dataset["valid"].with_transform(apply_val_transforms)
test_ds = dataset["test"].with_transform(apply_val_transforms)

## 3. EfficientNet-B4 Model Setup
We use HuggingFace's `AutoModelForImageClassification` to ensure 100% compatibility with `analyzer.py`.

In [ ]:
from transformers import AutoModelForImageClassification, AutoImageProcessor

MODEL_ID = "google/efficientnet-b4"

processor = AutoImageProcessor.from_pretrained(MODEL_ID)
model = AutoModelForImageClassification.from_pretrained(
    MODEL_ID,
    num_labels=2,
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True,
)

print("EfficientNet-B4 loaded successfully!")

## 4. Evaluation Metrics & Training Configuration

In [ ]:
import evaluate
from transformers import TrainingArguments, Trainer

accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")
precision_metric = evaluate.load("precision")
recall_metric = evaluate.load("recall")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    preds = np.argmax(predictions, axis=1)
    return {
        "accuracy": accuracy_metric.compute(predictions=preds, references=labels)["accuracy"],
        "f1": f1_metric.compute(predictions=preds, references=labels, average="weighted")["f1"],
        "precision": precision_metric.compute(predictions=preds, references=labels, average="weighted")["precision"],
        "recall": recall_metric.compute(predictions=preds, references=labels, average="weighted")["recall"],
    }

def collate_fn(examples):
    pixel_values = torch.stack([example["pixel_values"] for example in examples])
    labels = torch.tensor([example["label"] for example in examples])
    return {"pixel_values": pixel_values, "labels": labels}

OUTPUT_DIR = "./deeptrace-efficientnet"

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=1e-4,              # Slightly higher LR for head fine-tuning
    per_device_train_batch_size=16,  # B4 is memory intensive
    gradient_accumulation_steps=2,   # Effective batch size = 32
    per_device_eval_batch_size=32,
    num_train_epochs=5,              # Start with 5 for demonstration
    warmup_ratio=0.1,
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    save_total_limit=1,
    fp16=torch.cuda.is_available(),  # Mixed precision (fast)
    label_smoothing_factor=0.1,      # Prevents overconfidence
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    processing_class=processor,
    data_collator=collate_fn,
    compute_metrics=compute_metrics,
)

In [ ]:
# 🚀 Run Training
print("Starting EfficientNet-B4 fine-tuning...")
trainer.train()

In [ ]:
# 📊 Evaluate on Test Set
print("Evaluating on test set...")
metrics = trainer.evaluate(test_ds)
print(metrics)

In [ ]:
# 💾 Save and Export Model
import shutil
from google.colab import files

trainer.save_model(OUTPUT_DIR)
processor.save_pretrained(OUTPUT_DIR)

zip_filename = "deeptrace_efficientnet.zip"
shutil.make_archive("deeptrace_efficientnet", "zip", OUTPUT_DIR)

print(f"Created {zip_filename}. Downloading to local machine...")
files.download(zip_filename)
print("Extract this zip into backend/models/deeptrace-efficientnet and update your .env file!")